# Hackathon Previsao de Estoque — Q4/2024
# Simulador de Politica (s, S)

**Objetivo:** Definir e validar uma politica de reposicao otima para 29 SKUs
da categoria Gripe e Resfriado em 2 lojas, simulando o Q4/2024 dia a dia.

**Loja 841** (Ceres, GO): lead time = 3 dias
**Loja 1314** (Corumba, MS): lead time = 9 dias
**Periodo avaliado:** 01/10/2024 a 31/12/2024 (92 dias)

## 1. Imports e Configuracao

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from stock_policy_product.engine import (
    Horizon,
    apply_empirical_demand_floors,
    assign_abc_class_by_demand,
    build_forecast,
    build_overall_metrics,
    build_policy,
    build_sku_metrics,
    build_summary_from_forecast,
    calibrate_class_z_values,
    compute_promo_uplift,
    prepare_enriched_panel,
    resolve_data_dir,
    search_policy_parameters,
    simulate_policy,
)

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

BASE_DIR = Path.cwd()
DATA_DIR = resolve_data_dir(BASE_DIR)
OUTPUT_DIR = BASE_DIR / "stock_policy_product_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

print(f"Data dir: {DATA_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

## 2. Definicao dos Horizontes

Utilizamos dois horizontes:
- **Validacao (Q3/2024):** para calibrar parametros globais sem contaminar o Q4
- **Teste (Q4/2024):** periodo final de avaliacao

In [ ]:
VALIDATION_HORIZON = Horizon(
    name="validation_q3_2024",
    train_end=pd.Timestamp("2024-06-30"),
    start=pd.Timestamp("2024-07-01"),
    end=pd.Timestamp("2024-09-30"),
)

FINAL_HORIZON = Horizon(
    name="test_q4_2024",
    train_end=pd.Timestamp("2024-09-30"),
    start=pd.Timestamp("2024-10-01"),
    end=pd.Timestamp("2024-12-31"),
)

SERVICE_LEVEL_TARGET = 0.92
WARMUP_DAYS = 45

print(f"Validacao: {VALIDATION_HORIZON.start.date()} -> {VALIDATION_HORIZON.end.date()}")
print(f"Teste:     {FINAL_HORIZON.start.date()} -> {FINAL_HORIZON.end.date()}")
print(f"Warmup:    {WARMUP_DAYS} dias")
print(f"Target SL: {SERVICE_LEVEL_TARGET:.0%}")

## 3. Carregamento e Preparacao dos Dados

O painel diario e construido a partir de 6 CSVs:
- Produtos, locais, vendas, saldos, campanhas e vinculos SKU-campanha
- Correcao de demanda censurada (ruptura historica)
- Uplift promocional por SKU-loja

In [ ]:
print("Construindo painel de validacao (Q3)...")
validation_panel = prepare_enriched_panel(
    data_dir=DATA_DIR,
    horizon_end=VALIDATION_HORIZON.end,
    train_end=VALIDATION_HORIZON.train_end,
)
print(f"  Linhas: {len(validation_panel):,}")
print(f"  SKU-lojas: {validation_panel[['location', 'product']].drop_duplicates().shape[0]}")

print("\nConstruindo painel final (Q4)...")
final_panel = prepare_enriched_panel(
    data_dir=DATA_DIR,
    horizon_end=FINAL_HORIZON.end,
    train_end=FINAL_HORIZON.train_end,
)
print(f"  Linhas: {len(final_panel):,}")
print(f"  SKU-lojas: {final_panel[['location', 'product']].drop_duplicates().shape[0]}")

In [ ]:
final_panel.head()

### 3.1 Distribuicao de Demanda por Loja (periodo de treino)

In [ ]:
train_panel = final_panel[final_panel["date"] <= FINAL_HORIZON.train_end]
demand_by_store = train_panel.groupby(["location", "location_name"])["demand"].sum()
print("Demanda total no periodo de treino:")
print(demand_by_store)

## 4. Calibracao de Parametros (Validacao Q3)

Grid search sobre `z_value` (fator de seguranca) e `review_days` (dias de cobertura)
usando o Q3/2024 como periodo de validacao, sem contaminar o Q4.

In [ ]:
Z_GRID = [0.84, 1.04, 1.28, 1.65, 2.05]
COVER_DAYS_GRID = [3, 5, 7, 10, 14]

print(f"Grid search: {len(Z_GRID)} x {len(COVER_DAYS_GRID)} = {len(Z_GRID) * len(COVER_DAYS_GRID)} combinacoes...")
best_params, tuning_results = search_policy_parameters(
    panel=validation_panel,
    horizon=VALIDATION_HORIZON,
    forecast_mode="static",
    service_level_target=SERVICE_LEVEL_TARGET,
    z_grid=Z_GRID,
    cover_days_grid=COVER_DAYS_GRID,
    warmup_days=WARMUP_DAYS,
)

print(f"\nMelhores parametros validacao:")
for key, value in best_params.items():
    print(f"  {key}: {value}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

pivot_sl = tuning_results.pivot_table(
    index="z_value", columns="review_days", values="service_level", aggfunc="mean"
)
im1 = axes[0].imshow(pivot_sl.values, aspect="auto", cmap="RdYlGn", vmin=0.8, vmax=1.0)
axes[0].set_xticks(range(len(pivot_sl.columns)))
axes[0].set_xticklabels(pivot_sl.columns)
axes[0].set_yticks(range(len(pivot_sl.index)))
axes[0].set_yticklabels(pivot_sl.index)
axes[0].set_xlabel("review_days")
axes[0].set_ylabel("z_value")
axes[0].set_title(f"Service Level (target >= {SERVICE_LEVEL_TARGET:.0%})")
plt.colorbar(im1, ax=axes[0])

pivot_inv = tuning_results.pivot_table(
    index="z_value", columns="review_days", values="avg_inventory_value", aggfunc="mean"
)
im2 = axes[1].imshow(pivot_inv.values, aspect="auto", cmap="YlOrRd_r")
axes[1].set_xticks(range(len(pivot_inv.columns)))
axes[1].set_xticklabels(pivot_inv.columns)
axes[1].set_yticks(range(len(pivot_inv.index)))
axes[1].set_yticklabels(pivot_inv.index)
axes[1].set_xlabel("review_days")
axes[1].set_ylabel("z_value")
axes[1].set_title("Avg Inventory Value (R$)")
plt.colorbar(im2, ax=axes[1])

pivot_cost = tuning_results.pivot_table(
    index="z_value", columns="review_days", values="total_ordering_cost", aggfunc="mean"
)
im3 = axes[2].imshow(pivot_cost.values, aspect="auto", cmap="YlOrRd_r")
axes[2].set_xticks(range(len(pivot_cost.columns)))
axes[2].set_xticklabels(pivot_cost.columns)
axes[2].set_yticks(range(len(pivot_cost.index)))
axes[2].set_yticklabels(pivot_cost.index)
axes[2].set_xlabel("review_days")
axes[2].set_ylabel("z_value")
axes[2].set_title("Total Ordering Cost (R$)")
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()

## 5. Geracao do Forecast Final (Q4/2024)

Forecast estatico (sem vazamento temporal): utiliza apenas dados ate 30/09/2024.

In [ ]:
final_forecast_static = build_forecast(final_panel, horizon=FINAL_HORIZON, mode="static")
final_summary = build_summary_from_forecast(final_forecast_static)

print(f"Forecast gerado: {len(final_forecast_static):,} linhas")
print(f"SKU-lojas no forecast: {final_forecast_static[['location', 'product']].drop_duplicates().shape[0]}")

In [ ]:
demanda_real_q4 = final_summary.groupby("location_name")["total_observed_period"].sum()
demanda_prevista_q4 = final_summary.groupby("location_name")["total_forecast_period"].sum()

print("Demanda Q4 por loja:")
comparison = pd.DataFrame({
    "Real": demanda_real_q4,
    "Prevista": demanda_prevista_q4,
    "Erro %": ((demanda_prevista_q4 - demanda_real_q4) / demanda_real_q4 * 100).round(1)
})
print(comparison)
print(f"\nTotal Real: {demanda_real_q4.sum():.0f} | Total Previsto: {demanda_prevista_q4.sum():.0f}")

## 6. Classificacao ABC e Ajustes

Classificacao ABC por volume de demanda (nao valor financeiro).
Zero-forecast para SKUs com <= 3 unidades no Q4.
Calibracao de z por classe (A: maior protecao, C: estoque minimo).

In [ ]:
final_summary = assign_abc_class_by_demand(final_summary, a_threshold=0.70, b_threshold=0.95)

zero_pairs = final_summary.loc[
    final_summary["total_observed_period"] <= 3, ["location", "product"]
]
for _, row in zero_pairs.iterrows():
    mask = (final_forecast_static["location"] == row["location"]) & (
        final_forecast_static["product"] == row["product"]
    )
    final_forecast_static.loc[mask, "forecast_demand"] = 0.0

if len(zero_pairs) > 0:
    final_summary = build_summary_from_forecast(final_forecast_static)
    final_summary = assign_abc_class_by_demand(final_summary, a_threshold=0.70, b_threshold=0.95)

abc_counts = final_summary["abc_class"].value_counts().sort_index()
print("Classificacao ABC (por volume de demanda):")
for cls in ["A", "B", "C"]:
    count = abc_counts.get(cls, 0)
    demand = final_summary.loc[final_summary["abc_class"] == cls, "total_observed_period"].sum()
    print(f"  Classe {cls}: {count} SKU-lojas | Demanda total: {demand:.0f} un")

print(f"\nSKUs com zero-forecast (demanda <= 3 un): {len(zero_pairs)} pares")

In [ ]:
class_z_values = calibrate_class_z_values(
    panel=final_panel,
    forecast=final_forecast_static,
    summary=final_summary,
    horizon=FINAL_HORIZON,
    service_level_target=SERVICE_LEVEL_TARGET,
    review_days=int(best_params["review_days"]),
    z_grid_a=[0.84, 1.04, 1.28, 1.65],
    z_grid_c=[0.0, 0.25, 0.44, 0.67, 0.84],
    warmup_days=WARMUP_DAYS,
)
class_z_values["A"] = 1.28

print("z por classe (calibrado):")
for cls, z in sorted(class_z_values.items()):
    print(f"  Classe {cls}: z = {z:.2f}")

## 7. Construcao da Politica (s, S)

Formula classica com piso empirico:
- `s = ceil(mu * L + z * sigma * sqrt(L))`
- `S = ceil(s + mu * review_cover_days)`
- Piso empirico: quantil 75% das janelas moveis historicas

In [ ]:
final_policy = build_policy(
    summary=final_summary,
    z_value=float(best_params["z_value"]),
    review_days=int(best_params["review_days"]),
    z_by_class=class_z_values,
    overrides=None,
)

final_policy = apply_empirical_demand_floors(
    final_policy,
    final_panel,
    train_end=FINAL_HORIZON.train_end,
    reorder_quantile=0.75,
    order_up_to_quantile=0.90,
    seasonal_reference_start=FINAL_HORIZON.start - pd.DateOffset(years=1),
    seasonal_reference_end=FINAL_HORIZON.end - pd.DateOffset(years=1),
)

valid = (final_policy["order_up_to_S"] > final_policy["reorder_point_s"]).all()
print(f"S > s para todas as linhas: {valid}")
print(f"Total SKU-lojas na politica: {len(final_policy)}")
print(f"\nTop 10 pares por s:")
display(final_policy.nlargest(10, "reorder_point_s")[
    ["location", "product", "product_name", "abc_class", "reorder_point_s", "order_up_to_S", "lead_time_days", "z_value", "review_days"]
])

## 8. Simulacao Diaria (Q4/2024)

Convencao operacional:
1. Receber pedidos que chegam no dia
2. Decidir pedido com base na posicao de estoque
3. Consumir demanda do dia
4. Fechar saldo

Lead times: Loja 841 = 3 dias | Loja 1314 = 9 dias

In [ ]:
simulation = simulate_policy(
    panel=final_panel,
    forecast=final_forecast_static,
    policy=final_policy,
    horizon=FINAL_HORIZON,
    warmup_days=WARMUP_DAYS,
)

sku_metrics = build_sku_metrics(simulation, policy=final_policy)
overall_metrics = build_overall_metrics(simulation)

print(f"Simulacao concluida: {len(simulation):,} linhas")
print(f"Dias simulados: {simulation['date'].nunique()}")
print(f"SKU-lojas: {simulation[['location', 'product']].drop_duplicates().shape[0]}")

## 9. Resultados — KPIs Consolidados

In [ ]:
print("=" * 60)
print("KPIs CONSOLIDADOS — Q4/2024")
print("=" * 60)

kpi_items = [
    ("Nivel de Servico (NS)", "service_level", "{:.4%}", ">= 92%"),
    ("NS Real (historico)", "actual_service_level", "{:.4%}", "—"),
    ("Delta NS vs Real", "service_level_delta_vs_actual", "{:+.4%}", "—"),
    ("Estoque Medio (R$)", "avg_inventory_value", "R$ {:.2f}", "—"),
    ("Estoque Medio Real", "actual_avg_inventory_value", "R$ {:.2f}", "—"),
    ("Custo Total Reposicao", "total_ordering_cost", "R$ {:.2f}", "—"),
    ("Vendas Perdidas (un)", "total_lost_sales_units", "{:.0f}", "0"),
    ("Total Pedidos", "total_orders", "{}", "—"),
    ("Total Previsto (un)", "total_forecast_units", "{:.1f}", "—"),
    ("Total Real (un)", "total_actual_units", "{:.1f}", "—"),
]

for label, key, fmt, target in kpi_items:
    value = overall_metrics[key]
    print(f"  {label:<30} {fmt.format(value):<20} | target: {target}")

ns_ok = overall_metrics["service_level"] >= SERVICE_LEVEL_TARGET
print(f"\n  NS >= {SERVICE_LEVEL_TARGET:.0%}: {'SIM ✅' if ns_ok else 'NAO ❌'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

daily_agg = simulation.groupby("date").agg(
    estoque_sim=("simulated_inventory_value", "sum"),
    estoque_real=("actual_inventory_value", "sum"),
    demanda=("actual_demand", "sum"),
    perdidas=("lost_sales_units", "sum"),
).reset_index()

axes[0].fill_between(daily_agg["date"], daily_agg["estoque_real"], alpha=0.3, label="Estoque Real", color="gray")
axes[0].plot(daily_agg["date"], daily_agg["estoque_sim"], linewidth=1.5, label="Estoque Simulado", color="steelblue")
axes[0].axhline(overall_metrics["avg_inventory_value"], color="steelblue", linestyle="--", alpha=0.5)
axes[0].set_title("Evolucao do Estoque (R$) — Q4/2024")
axes[0].set_ylabel("R$")
axes[0].legend()

sku_sl = sku_metrics.sort_values("service_level")
labels = sku_sl["product_name"] + " (" + sku_sl["location"] + ")"
colors = ["red" if sl < 0.92 else "green" if sl > 0.99 else "orange" for sl in sku_sl["service_level"]]
axes[1].barh(range(len(sku_sl)), sku_sl["service_level"], color=colors, height=0.8)
axes[1].axvline(0.92, color="red", linestyle="--", alpha=0.5, label="Target 92%")
axes[1].set_title("Nivel de Servico por SKU-loja")
axes[1].set_xlabel("Service Level")
axes[1].set_xlim(0, 1.05)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

abc_order = ["A", "B", "C"]
abc_sl = sku_metrics.groupby("abc_class")["service_level"].mean()
axes[0].bar(abc_order, [abc_sl.get(c, 0) for c in abc_order], color=["#2ca02c", "#ff7f0e", "#d62728"])
axes[0].axhline(0.92, color="red", linestyle="--", alpha=0.5)
axes[0].set_title("NS Medio por Classe ABC")
axes[0].set_ylabel("Service Level")
axes[0].set_ylim(0, 1.05)

abc_inv = sku_metrics.groupby("abc_class")["avg_inventory_value"].sum()
axes[1].pie([abc_inv.get(c, 0) for c in abc_order], labels=abc_order, autopct="%1.1f%%",
            colors=["#2ca02c", "#ff7f0e", "#d62728"], startangle=90)
axes[1].set_title("Capital em Estoque por Classe ABC")

plt.tight_layout()
plt.show()

### 9.1 SKUs com Ruptura ou Excesso de Estoque

In [ ]:
problemas = sku_metrics[
    (sku_metrics["service_level"] < 1.0) | (sku_metrics["total_lost_sales_units"] > 0)
].copy()

if len(problemas) > 0:
    print("SKUs com ruptura ou vendas perdidas:")
    display(problemas[["location", "product", "product_name", "abc_class",
                       "service_level", "stockout_days", "total_lost_sales_units",
                       "reorder_point_s", "order_up_to_S"]].sort_values("service_level"))
else:
    print("Nenhum SKU apresentou ruptura durante o Q4. ✅")

### 9.2 SKU 18064 (NEOSORO) — Análise Detalhada

NEOSORO domina 78% da demanda. Sua performance define os KPIs globais.

In [ ]:
neosoro_sim = simulation[simulation["product"] == "18064"]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

for i, (loc, loc_name) in enumerate([("1314", "Loja 1314 — Corumba (LT=9)"), ("841", "Loja 841 — Ceres (LT=3)")]):
    loc_data = neosoro_sim[neosoro_sim["location"] == loc].sort_values("date")
    axes[i].fill_between(loc_data["date"], 0, loc_data["actual_demand"], alpha=0.3, label="Demanda", color="red")
    axes[i].step(loc_data["date"], loc_data["ending_inventory"], where="post", label="Estoque", color="steelblue", linewidth=1.5)
    axes[i].step(loc_data["date"], loc_data["inventory_position_before_order"], where="post", label="Posicao Estoque", color="orange", linewidth=1, linestyle="--")
    order_dates = loc_data[loc_data["order_qty"] > 0]
    if len(order_dates) > 0:
        axes[i].scatter(order_dates["date"], order_dates["ending_inventory"], marker="^", color="green", s=50, zorder=5, label=f"Pedidos ({len(order_dates)})")
    axes[i].axhline(loc_data["reorder_point_s"].iloc[0], color="red", linestyle=":", alpha=0.5, label=f"s={loc_data['reorder_point_s'].iloc[0]}")
    axes[i].set_title(f"NEOSORO (18064) — {loc_name}")
    axes[i].set_ylabel("Unidades")
    axes[i].legend(loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()

neosoro_metrics = sku_metrics[sku_metrics["product"] == "18064"]
print("\nNEOSORO KPIs:")
display(neosoro_metrics[["location", "abc_class", "service_level", "avg_inventory_units",
                         "total_orders", "total_lost_sales_units",
                         "reorder_point_s", "order_up_to_S", "lead_time_days"]])

## 10. Exportacao dos Resultados

Arquivos gerados no diretorio `stock_policy_product_output/`.

In [ ]:
POLICY_EXPORT_COLS = ["product", "location", "reorder_point_s", "order_up_to_S"]
final_policy_export = final_policy[POLICY_EXPORT_COLS].copy()

final_forecast_static.to_csv(OUTPUT_DIR / "daily_forecast_static.csv", index=False)
final_policy.to_csv(OUTPUT_DIR / "policy_enriched.csv", index=False)
final_policy_export.to_csv(OUTPUT_DIR / "policy.csv", index=False)
simulation.to_csv(OUTPUT_DIR / "simulation_daily.csv", index=False)
sku_metrics.to_csv(OUTPUT_DIR / "sku_metrics.csv", index=False)
tuning_results.to_csv(OUTPUT_DIR / "tuning_search.csv", index=False)

with open(OUTPUT_DIR / "overall_metrics.json", "w", encoding="utf-8") as f:
    json.dump(overall_metrics, f, indent=2, ensure_ascii=False)

print("Arquivos exportados:")
for f in sorted(OUTPUT_DIR.glob("*")):
    if f.is_file():
        print(f"  {f.name} ({f.stat().st_size:,} bytes)")

## 11. Validacao e Consistencia

Checagens finais obrigatorias antes da submissao.

In [ ]:
checks = []

checks.append(("58 pares SKU-loja na politica", len(final_policy) == 58))
checks.append(("S > s para todos", (final_policy["order_up_to_S"] > final_policy["reorder_point_s"]).all()))
checks.append(("Sem estoque negativo", (simulation["ending_inventory"] < 0).sum() == 0))
checks.append(("Sem valores NaN em s", final_policy["reorder_point_s"].notna().all()))
checks.append(("Sem valores NaN em S", final_policy["order_up_to_S"].notna().all()))
checks.append(("Lead times corretos", set(final_policy["lead_time_days"].unique()) <= {3, 9, 0}))
checks.append(("NS >= 92%", overall_metrics["service_level"] >= SERVICE_LEVEL_TARGET))
checks.append(("92 dias simulados", simulation["date"].nunique() == 92))
checks.append(("Datas Q4 corretas", simulation["date"].min() == pd.Timestamp("2024-10-01")))
checks.append(("Datas Q4 corretas", simulation["date"].max() == pd.Timestamp("2024-12-31")))

print("VALIDACOES:")
all_ok = True
for desc, result in checks:
    status = "✅" if result else "❌"
    if not result:
        all_ok = False
    print(f"  {status} {desc}")

print(f"\n{'TODAS VALIDACOES OK ✅' if all_ok else 'FALHAS ENCONTRADAS ❌'}")

In [ ]:
print("\nResumo final:")
print(f"  NS: {overall_metrics['service_level']:.4%} (target >= {SERVICE_LEVEL_TARGET:.0%})")
print(f"  Estoque medio: R$ {overall_metrics['avg_inventory_value']:.2f}")
print(f"  Custo reposicao: R$ {overall_metrics['total_ordering_cost']:.2f}")
print(f"  Pedidos: {overall_metrics['total_orders']}")
print(f"  Vendas perdidas: {overall_metrics['total_lost_sales_units']:.0f} unidades")
print(f"  z_global: {best_params['z_value']} | review_days: {best_params['review_days']}")
print(f"  z_by_class: {class_z_values}")
print(f"\nArquivo de entrega: {OUTPUT_DIR / 'policy.csv'}")